# StudyAssistant v2.0
**Maharashtra SSC Board — Science 1 (Class 9) | Chapters 1 & 2**

End-to-end RAG pipeline: Ingestion → Embeddings → Generation → Evaluation → Fix

---

## Stage 1 — Ingestion & Chunking

In [ ]:
import os, re, json
import fitz  # PyMuPDF
import tiktoken
from rank_bm25 import BM25Okapi
from dotenv import load_dotenv

load_dotenv()

PDF_PATH = "Science_1_SSC_Testbook.pdf"
CHUNKS_FILE = "wk10_chunks.json"
ENC = tiktoken.get_encoding("cl100k_base")
MAX_TOKENS = 250

print("Imports OK")

In [ ]:
def load_pdf_pages(pdf_path: str) -> list[dict]:
    """Extract text per page using PyMuPDF."""
    doc = fitz.open(pdf_path)
    pages = []
    for i, page in enumerate(doc):
        text = page.get_text("text")
        pages.append({"page": i + 1, "text": text})
    doc.close()
    print(f"Loaded {len(pages)} pages from {pdf_path}")
    return pages

all_pages = load_pdf_pages(PDF_PATH)

# Preview first page
print("\n--- Page 1 preview (first 500 chars) ---")
print(all_pages[0]["text"][:500])

In [ ]:
def find_chapter_pages(pages: list[dict], chapter_nums: list[int]) -> list[dict]:
    """
    Detect start/end pages for each chapter.
    Looks for lines like '1. Laws of Motion' or 'Chapter 1' or '2. Work and Energy'.
    Returns pages belonging to the requested chapters.
    """
    # Patterns that signal a chapter heading
    chapter_start_pattern = re.compile(
        r"^\s*(Chapter\s+\d+|\d+\.\s+[A-Z])",
        re.MULTILINE | re.IGNORECASE
    )

    chapter_map = {}  # chapter_num -> start_page_index
    for i, p in enumerate(pages):
        for line in p["text"].splitlines():
            line = line.strip()
            # Match "1." or "Chapter 1" at line start
            m = re.match(r"^(\d+)\s*[.:]\s+[A-Z]", line)
            if m:
                num = int(m.group(1))
                if num not in chapter_map and 1 <= num <= 20:
                    chapter_map[num] = i
                    break

    print("Detected chapter starts (page index):", chapter_map)

    # Collect pages for requested chapters
    selected = []
    sorted_chapters = sorted(chapter_map.keys())

    for ch in chapter_nums:
        if ch not in chapter_map:
            print(f"WARNING: Chapter {ch} not detected — including heuristic range")
            continue
        start = chapter_map[ch]
        # end = start of next chapter OR 50 pages later
        ch_index = sorted_chapters.index(ch)
        if ch_index + 1 < len(sorted_chapters):
            end = chapter_map[sorted_chapters[ch_index + 1]]
        else:
            end = min(start + 50, len(pages))
        for p in pages[start:end]:
            selected.append({**p, "chapter": ch})

    print(f"Selected {len(selected)} pages for chapters {chapter_nums}")
    return selected

chapter_pages = find_chapter_pages(all_pages, chapter_nums=[1, 2])

In [ ]:
def count_tokens(text: str) -> int:
    return len(ENC.encode(text))


def classify_content_type(text: str) -> str:
    """
    Classify a block of text into one of three types:
      - worked_example : contains 'Example' keyword (SSC textbook uses this heavily)
      - question_or_exercise : starts with Q. / Exercise / numbered question
      - prose : everything else
    """
    t = text.strip()
    if re.search(r"\bExample\b", t, re.IGNORECASE):
        return "worked_example"
    if re.match(r"^(Q\.?\s?\d+|Exercise|\d+\.\s+[A-Z].*\?)", t, re.IGNORECASE):
        return "question_or_exercise"
    if re.search(r"\b(Q\.?\d+|Exercise\s+\d+)", t, re.IGNORECASE):
        return "question_or_exercise"
    return "prose"


def extract_section(text: str) -> str:
    """Pull the first heading-like line from a block as section name."""
    for line in text.splitlines():
        line = line.strip()
        # Heading: short, starts with capital, no period mid-line
        if 3 < len(line) < 80 and line[0].isupper() and line.count('.') <= 1:
            return line
    return "Unknown"


def split_into_paragraphs(text: str) -> list[str]:
    """Split page text on blank lines; preserve multi-line paragraphs."""
    blocks = re.split(r"\n{2,}", text)
    return [b.strip() for b in blocks if b.strip()]


def chunk_pages(pages: list[dict], source: str) -> list[dict]:
    """
    Content-type-aware chunker:
    1. Split each page into paragraph blocks
    2. Classify each block
    3. Merge small consecutive prose blocks until <=250 tokens
    4. Never merge across worked_example or question_or_exercise boundaries
    """
    chunks = []
    chunk_id_counter = 0

    for page_info in pages:
        page_num = page_info["page"]
        chapter = page_info["chapter"]
        paragraphs = split_into_paragraphs(page_info["text"])

        buffer = ""
        buffer_type = "prose"

        def flush_buffer():
            nonlocal buffer, buffer_type, chunk_id_counter
            if not buffer.strip():
                return
            token_count = count_tokens(buffer)
            chunk = {
                "chunk_id": f"ch{chapter}_p{page_num}_{chunk_id_counter:04d}",
                "source": source,
                "section": extract_section(buffer),
                "content_type": buffer_type,
                "page": page_num,
                "chapter": chapter,
                "token_count": token_count,
                "content": buffer.strip(),
            }
            chunks.append(chunk)
            chunk_id_counter += 1
            buffer = ""
            buffer_type = "prose"

        for para in paragraphs:
            if not para:
                continue
            p_type = classify_content_type(para)
            p_tokens = count_tokens(para)

            # Special types always get their own chunk
            if p_type in ("worked_example", "question_or_exercise"):
                flush_buffer()
                # If this single para exceeds limit, hard-split it
                if p_tokens > MAX_TOKENS:
                    words = para.split()
                    sub = ""
                    for word in words:
                        candidate = (sub + " " + word).strip()
                        if count_tokens(candidate) > MAX_TOKENS:
                            buffer = sub
                            buffer_type = p_type
                            flush_buffer()
                            sub = word
                        else:
                            sub = candidate
                    if sub:
                        buffer = sub
                        buffer_type = p_type
                        flush_buffer()
                else:
                    buffer = para
                    buffer_type = p_type
                    flush_buffer()
                continue

            # Prose: accumulate until token limit
            candidate = (buffer + "\n\n" + para).strip() if buffer else para
            if count_tokens(candidate) > MAX_TOKENS:
                flush_buffer()
                buffer = para
                buffer_type = "prose"
            else:
                buffer = candidate
                buffer_type = "prose"

        flush_buffer()  # end of page

    return chunks


chunks = chunk_pages(chapter_pages, source="Science_1_SSC_Testbook.pdf")
print(f"Total chunks: {len(chunks)}")

# Content type distribution
from collections import Counter
dist = Counter(c["content_type"] for c in chunks)
print("Content type distribution:", dict(dist))

# Token stats
tokens = [c["token_count"] for c in chunks]
print(f"Token stats — min: {min(tokens)}, max: {max(tokens)}, avg: {sum(tokens)/len(tokens):.1f}")

In [ ]:
# Preview a sample of each type
for ctype in ["prose", "worked_example", "question_or_exercise"]:
    sample = next((c for c in chunks if c["content_type"] == ctype), None)
    if sample:
        print(f"\n=== {ctype} (chunk_id: {sample['chunk_id']}, page: {sample['page']}, tokens: {sample['token_count']}) ===")
        print(sample["content"][:400])
    else:
        print(f"\n=== {ctype}: NONE FOUND ===")

In [ ]:
with open(CHUNKS_FILE, "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

print(f"Saved {len(chunks)} chunks → {CHUNKS_FILE}")

In [ ]:
# BM25 lexical retrieval test
def build_bm25(chunks: list[dict]) -> BM25Okapi:
    tokenized = [c["content"].lower().split() for c in chunks]
    return BM25Okapi(tokenized)


def bm25_search(query: str, bm25: BM25Okapi, chunks: list[dict], k: int = 3) -> list[dict]:
    tokens = query.lower().split()
    scores = bm25.get_scores(tokens)
    top_idx = sorted(range(len(scores)), key=lambda i: scores[i], reverse=True)[:k]
    return [
        {"rank": r + 1, "chunk_id": chunks[i]["chunk_id"], "score": round(scores[i], 4),
         "page": chunks[i]["page"], "snippet": chunks[i]["content"][:200]}
        for r, i in enumerate(top_idx)
    ]


bm25 = build_bm25(chunks)

test_queries = [
    "What is the law of conservation of mass?",
    "Define atom and molecule",
    "What is the difference between element and compound?",
    "Explain physical and chemical change with example",
    "What is mixture? Give types of mixtures"
]

print("BM25 Top-3 Results for 5 Test Queries\n" + "="*50)
for q in test_queries:
    print(f"\nQuery: {q}")
    results = bm25_search(q, bm25, chunks, k=3)
    for r in results:
        print(f"  [{r['rank']}] {r['chunk_id']} (page {r['page']}, score {r['score']})")
        print(f"      {r['snippet'][:150]}...")